## Phase 1 - Preprocessing

In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", 200)

df = pd.read_csv("../data/raw/Suicide_Detection.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (232074, 3)
Columns: ['Unnamed: 0', 'text', 'class']


## Text Column Analysis

In [ ]:
print("Text column overview:")
print("Missing values:", df["text"].isna().sum())
print("Empty/blank strings:", (df["text"].astype(str).str.strip() == "").sum())

print("\nText length summary:")
print(df["text"].astype(str).str.len().describe())

print("\nSample text snippets:")

for idx, text in df["text"].head(3).items():
    snippet = str(text).replace("\n", " ")[:200]
    print(f"{idx}: {snippet}")

Text column overview:
Missing values: 0
Empty/blank strings: 0

Text length summary:
count    232074.000000
mean        689.639736
std        1156.334007
min           3.000000
25%         138.000000
50%         315.000000
75%         801.000000
max       40297.000000
Name: text, dtype: float64

Sample text snippets:
0: Ex Wife Threatening SuicideRecently I left my wife for good because she has cheated on me twice and lied to me so much that I have decided to refuse to go back to her. As of a few days ago, she began 
1: Am I weird I don't get affected by compliments if it's coming from someone I know irl but I feel really good when internet strangers do it
2: Finally 2020 is almost over... So I can never hear "2020 has been a bad year" ever again. I swear to fucking God it's so annoying


## Cleaning Rules

Define a deterministic text cleaning function that can be reuse in the notebook and later move into `src/preprocess.py`.

In [3]:
import re
import string


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


sample_text = df.loc[0, "text"]
print("Original:\n", sample_text[:500])
print("\nCleaned:\n", clean_text(sample_text)[:500])

Original:
 Ex Wife Threatening SuicideRecently I left my wife for good because she has cheated on me twice and lied to me so much that I have decided to refuse to go back to her. As of a few days ago, she began threatening suicide. I have tirelessly spent these paat few days talking her out of it and she keeps hesitating because she wants to believe I'll come back. I know a lot of people will threaten this in order to get their way, but what happens if she really does? What do I do and how am I supposed to

Cleaned:
 ex wife threatening suiciderecently i left my wife for good because she has cheated on me twice and lied to me so much that i have decided to refuse to go back to her as of a few days ago she began threatening suicide i have tirelessly spent these paat few days talking her out of it and she keeps hesitating because she wants to believe ill come back i know a lot of people will threaten this in order to get their way but what happens if she really does what do i do and how 

## Cleaning full dataset

Apply the simple cleaning function to the full dataset and save a cleaned version for the next stage.

In [4]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_clean = df.drop(columns=["Unnamed: 0"]).copy()
df_clean["cleaned_text"] = df_clean["text"].apply(clean_text)

cleaned_path = output_dir / "cleaned_data.csv"
df_clean.to_csv(cleaned_path, index=False)

print("Saved:", cleaned_path)
print("Shape:", df_clean.shape)
print("Columns:", df_clean.columns.tolist())

Saved: ../data/processed/cleaned_data.csv
Shape: (232074, 3)
Columns: ['text', 'class', 'cleaned_text']


## Validation

Inspecting before splitting

In [5]:
print("Cleaned data overview:")
print("Shape:", df_clean.shape)

print("Columns:", df_clean.columns.tolist())
print("\nSample cleaned rows:")

for idx, row in df_clean[["text", "cleaned_text", "class"]].head(3).iterrows():
    print(f"\nRow {idx}")
    print("Class:", row["class"])
    print("Original:", str(row["text"])[:200])
    print("Cleaned:", str(row["cleaned_text"])[:200])

Cleaned data overview:
Shape: (232074, 3)
Columns: ['text', 'class', 'cleaned_text']

Sample cleaned rows:

Row 0
Class: suicide
Original: Ex Wife Threatening SuicideRecently I left my wife for good because she has cheated on me twice and lied to me so much that I have decided to refuse to go back to her. As of a few days ago, she began 
Cleaned: ex wife threatening suiciderecently i left my wife for good because she has cheated on me twice and lied to me so much that i have decided to refuse to go back to her as of a few days ago she began th

Row 1
Class: non-suicide
Original: Am I weird I don't get affected by compliments if it's coming from someone I know irl but I feel really good when internet strangers do it
Cleaned: am i weird i dont get affected by compliments if its coming from someone i know irl but i feel really good when internet strangers do it

Row 2
Class: non-suicide
Original: Finally 2020 is almost over... So I can never hear "2020 has been a bad year" ever again. I s

## Stratified Split

Splitting for tranning, validation and testing

In [6]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(
    df_clean,
    test_size=0.1,
    random_state=42,
    stratify=df_clean["class"],
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1111111111,
    random_state=42,
    stratify=train_val_df["class"],
)

train_path = output_dir / "train_data.csv"
val_path = output_dir / "validation_data.csv"
test_path = output_dir / "test_data.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("\nSaved files:")
print(train_path)
print(val_path)
print(test_path)
print("\nTrain class counts:")
print(train_df["class"].value_counts().to_string())
print("\nValidation class counts:")
print(val_df["class"].value_counts().to_string())
print("\nTest class counts:")
print(test_df["class"].value_counts().to_string())

Train shape: (185658, 3)
Validation shape: (23208, 3)
Test shape: (23208, 3)

Saved files:
../data/processed/train_data.csv
../data/processed/validation_data.csv
../data/processed/test_data.csv

Train class counts:
class
suicide        92829
non-suicide    92829

Validation class counts:
class
suicide        11604
non-suicide    11604

Test class counts:
class
non-suicide    11604
suicide        11604
